# Seaborn: asking statistical questions with charts

A café table contains bills and customer groups. The reader wants to compare spending patterns, but group size, spread and aggregation can change the story. Start with the question each chart should answer and inspect the observations behind the statistical summary.

Matplotlib gives us control over a figure. Seaborn builds on it to connect table columns to colors, groups, and statistical summaries. Instead of manually splitting every group, we name the variables and choose the relationship we want to inspect.

Imagine a café recording orders: the bill, party size, service period, and waiting time. We will use **synthetic teaching data** to explore distributions, comparisons, and relationships. These values describe no actual business. The examples run offline after installation.

Run the cells in order. Familiarity with Pandas DataFrames and Matplotlib's Figure/Axes objects is helpful.

## 1. Install and import

If needed, run `%pip install "seaborn>=0.13" pandas numpy matplotlib` in a separate cell. Seaborn uses Matplotlib to draw charts; we still use Matplotlib for layout, labels, and export.

`set_theme()` changes plotting defaults for this Python session. A readable theme is a starting point, while the choice of chart comes from the question.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
print("Seaborn:", sns.__version__)

## 2. Make a reproducible table

Each row is one order. A seeded random generator makes the data repeatable. We deliberately generate larger bills for larger parties and longer waits for dinner; patterns in these charts reflect those construction rules.

| Column | Meaning | Type |
| --- | --- | --- |
| `period` | Lunch or dinner | Category |
| `party_size` | People in the order | Integer |
| `bill` | Total bill in dollars | Number |
| `wait_minutes` | Waiting time | Number |
| `day` | Service day | Category |

Keep this distinction clear: generated data is useful for learning an API, but it cannot establish facts about real customers.

In [ ]:
rng = np.random.default_rng(42)
n = 180
period = rng.choice(["Lunch", "Dinner"], size=n)
party_size = rng.integers(1, 6, size=n)
orders = pd.DataFrame({
    "period": period,
    "party_size": party_size,
    "bill": np.round(np.maximum(5, 12 * party_size + 7 * (period == "Dinner") + rng.normal(0, 5, n)), 2),
    "wait_minutes": np.round(np.maximum(1, 5 + 2 * party_size + 4 * (period == "Dinner") + rng.normal(0, 3, n)), 1),
    "day": rng.choice(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat"], size=n),
})
period_order = ["Lunch", "Dinner"]
period_palette = {"Lunch": "#0072B2", "Dinner": "#D55E00"}
print("Table shape:", orders.shape)
print(orders.head())
print("Missing values:", orders.isna().sum().to_dict())

## 3. Map columns to visual properties

`data=orders` supplies the table; `x` and `y` name columns. `hue` maps a variable to color, and `style` maps it to marker shape. Repeating period through both color and shape helps distinguish the groups.

A scatter plot retains one mark per order, although overlapping marks can hide observations. Transparency helps reveal overlap.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")
sns.scatterplot(data=orders, x="bill", y="wait_minutes", hue="period", style="period",
                hue_order=period_order, palette=period_palette, alpha=0.75, ax=ax)
ax.set(title="Bill and waiting time for each order", xlabel="Bill ($)", ylabel="Wait (minutes)")
plt.show()

## 4. Choose the function from the question

| Question | Function | What is displayed |
| --- | --- | --- |
| How are two numerical values related? | `scatterplot` | Paired observations |
| What does a distribution look like? | `histplot`, `ecdfplot` | Counts or cumulative proportions |
| How do groups differ in spread? | `boxplot`, `violinplot` | Distribution summaries or smoothed densities |
| How many rows belong to each group? | `countplot` | Counts |
| How do group means compare? | `barplot` | Estimates and optional uncertainty |
| What changes across ordered x values? | `lineplot` | Aggregated or individual sequences |
| How can we compare subsets side by side? | `relplot`, `catplot`, `displot` | Faceted figures |

Counts, means, and raw observations answer different questions. Read the vertical axis before interpreting a chart.

## 5. Check how much the bins influence the apparent pattern

Predict how a different bin width would change the display. Compare more than one setting and describe stable features. A smooth-looking or sharply peaked histogram is partly a display choice.

These bins contain bill amounts grouped into $10 intervals. `multiple="dodge"` puts the two groups side by side within each bin. Counts depend on group size as well as distribution shape, so inspect how many orders each period contains.

In [ ]:
print(orders["period"].value_counts())
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
sns.histplot(data=orders, x="bill", hue="period", hue_order=period_order,
             palette=period_palette, binwidth=10, multiple="dodge", shrink=0.9, ax=ax)
ax.set(title="How frequently do bill amounts occur?", xlabel="Bill ($)", ylabel="Number of orders")
plt.show()

## 6. ECDFs avoid choosing bins

An empirical cumulative distribution function shows the fraction of observations at or below a value. A curve reaching 0.8 at $50 means 80% of that group's bills are at most $50. Each period is normalized separately, making proportions easier to compare when group sizes differ.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
sns.ecdfplot(data=orders, x="bill", hue="period", hue_order=period_order,
             palette=period_palette, ax=ax)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set(title="What fraction of bills falls below each amount?", xlabel="Bill ($)", ylabel="Cumulative proportion")
plt.show()

## 7. Box plots compare spread

The middle line is the median; the box spans the middle 50% of observations. Default whiskers extend to the furthest observations within 1.5 interquartile ranges of the box. Points beyond them are not necessarily errors.

Overlay the observations with `stripplot` to show sample size and overlap. Here `jitter=False` keeps the illustration deterministic; transparency reveals some overlapping points.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), layout="constrained")
sns.boxplot(data=orders, x="period", y="wait_minutes", order=period_order,
            color="lightgray", showfliers=False, ax=ax)
sns.stripplot(data=orders, x="period", y="wait_minutes", order=period_order,
              color="tab:blue", jitter=False, alpha=0.35, size=3, ax=ax)
ax.set(title="Waiting-time spread and individual orders", xlabel="Service period", ylabel="Wait (minutes)")
plt.show()

## 8. Violin plots use a smoothed estimate

A violin's width represents estimated density at a value. The quartile markings summarize location and spread. The shape depends on smoothing, so thin bumps are not automatically meaningful subgroups.

`cut=0` limits the drawing to the observed range. This limits the display; it does not remove all boundary effects from density estimation.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), layout="constrained")
sns.violinplot(data=orders, x="period", y="bill", order=period_order,
               hue="period", hue_order=period_order, palette=period_palette,
               inner="quart", cut=0, legend=False, ax=ax)
ax.set(title="Shape of bill distributions", xlabel="Service period", ylabel="Bill ($)")
plt.show()

## 9. Separate popularity from average spending

Before drawing, decide whether the bar height should count visits or summarize a numerical value. Compare the chart with the corresponding table calculation. A group can have more visits and a lower mean bill at the same time.

`countplot` counts rows. `barplot` calculates a mean by default. In the second panel, `errorbar=("ci", 95)` adds a bootstrap 95% confidence interval for each mean; it is not the range containing 95% of individual bills.

The bootstrap treats rows as independent observations. This synthetic example demonstrates the calculation; real repeated-customer data would require attention to dependence. The fixed bootstrap seed makes the interval repeatable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
sns.countplot(data=orders, x="period", order=period_order, color="tab:blue", ax=axes[0])
axes[0].set(title="How many orders?", xlabel="Service period", ylabel="Order count")
sns.barplot(data=orders, x="period", y="bill", order=period_order,
            color="tab:blue", errorbar=("ci", 95), n_boot=1000, seed=42, ax=axes[1])
axes[1].set(title="What is the mean bill?", xlabel="Service period", ylabel="Mean bill ($)")
plt.show()
print(orders.groupby("period")["bill"].agg(["size", "mean", "median"]))

## 10. Notice when one line point summarizes many rows

Count observations sharing an x value before interpreting the plotted point. Identify the estimator and uncertainty display. Connecting means is different from showing an individual customer’s path.

Several orders have the same party size. `lineplot` therefore estimates the mean bill at each size within each period. We explicitly disable intervals to focus on the aggregation. The connecting line describes ordered party sizes, not a customer's history over time.

For individual trajectories in longitudinal data, use an identifier through `units` and `estimator=None`; merely turning off error bars does not turn off averaging.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
sns.lineplot(data=orders, x="party_size", y="bill", hue="period", style="period",
             hue_order=period_order, palette=period_palette, markers=True,
             estimator="mean", errorbar=None, ax=ax)
ax.set(title="Mean bill by party size", xlabel="People in party", ylabel="Mean bill ($)")
ax.set_xticks(range(1, 6))
plt.show()

## 11. Axes-level and figure-level functions

Functions such as `scatterplot` draw on an existing Matplotlib Axes and accept `ax=`. Functions such as `relplot` create and manage their own figure, including panels selected by `col` or `row`.

Use an axes-level function inside a custom Matplotlib layout. Use a figure-level function when you want Seaborn to arrange comparable subsets. For a figure-level plot, `height` is each panel's height in inches and `aspect` is its width divided by height.

In [ ]:
grid = sns.relplot(data=orders, x="bill", y="wait_minutes", col="period",
                   col_order=period_order, kind="scatter", height=4, aspect=1,
                   alpha=0.65)
grid.set_axis_labels("Bill ($)", "Wait (minutes)")
grid.set_titles("{col_name} orders")
plt.show()

## 12. Treat a fitted relationship as an association to investigate

Predict the direction of the fitted trend, then inspect scatter and subgroup structure. A regression line does not establish that changing one variable will cause the other to change. Ask what unmeasured differences could explain the association.

`regplot` draws observations and a fitted line. Here we disable its interval with `ci=None` and fit within lunch orders only. A slope does not establish that paying more causes waiting longer: party size influenced both quantities when we generated the data.

A fitted straight line also does not guarantee that a linear model is suitable for another dataset.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
sns.regplot(data=orders.loc[orders["period"] == "Lunch"], x="bill", y="wait_minutes",
            ci=None, scatter_kws={"alpha": 0.5}, line_kws={"color": "black"}, ax=ax)
ax.set(title="Linear fit within lunch orders", xlabel="Bill ($)", ylabel="Wait (minutes)")
plt.show()

## 13. Correlation heatmaps summarize numerical relationships

Pearson correlation measures linear association on a scale from −1 to 1. Select meaningful numerical columns explicitly: encoding categories as arbitrary integers does not make their correlations meaningful.

A diverging palette centered at zero distinguishes positive and negative values. Correlation can miss nonlinear relationships and does not establish causation.

In [ ]:
correlations = orders[["party_size", "bill", "wait_minutes"]].corr()
fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
sns.heatmap(correlations, annot=True, fmt=".2f", cmap="vlag", center=0,
            vmin=-1, vmax=1, square=True, cbar_kws={"label": "Pearson correlation"}, ax=ax)
ax.set_title("Linear associations in the generated data")
plt.show()

## 14. Reshape wide data into long data

Seaborn often works naturally with long data: one row per observation, with separate columns naming the measurement and its value. Pandas `melt` turns several measurement columns into those two columns.

Bills and waiting times have different units. We put them in separate panels with independent x scales; combining them into one shared numerical distribution would be misleading.

In [ ]:
long_orders = orders.melt(id_vars=["period"], value_vars=["bill", "wait_minutes"],
                          var_name="measurement", value_name="value")
print(long_orders.head())
grid = sns.displot(data=long_orders, x="value", hue="period", col="measurement",
                   palette=period_palette, hue_order=period_order,
                   bins=15, height=4, facet_kws={"sharex": False, "sharey": False})
for name, ax in grid.axes_dict.items():
    ax.set_xlabel({"bill": "Bill ($)", "wait_minutes": "Wait (minutes)"}[name])
grid.set_titles("{col_name}")
plt.show()

## 15. Complete exercise: a café report

Combine the relationship, distribution, mean comparison, and order counts in one figure. Keep the service-period colors consistent across panels. Put weekdays in calendar order instead of whichever order appears first in the table.

Running this cell creates `outputs` and writes PNG and SVG reports, replacing those named files if they exist. Save through the Figure object before showing it.

In [ ]:
day_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
with sns.axes_style("whitegrid"):
    report, axes = plt.subplots(2, 2, figsize=(12, 9), layout="constrained")
    sns.scatterplot(data=orders, x="bill", y="wait_minutes", hue="period", style="period",
                    palette=period_palette, hue_order=period_order, alpha=0.65, ax=axes[0, 0])
    axes[0, 0].set(title="Individual orders", xlabel="Bill ($)", ylabel="Wait (minutes)")
    sns.ecdfplot(data=orders, x="wait_minutes", hue="period", palette=period_palette,
                 hue_order=period_order, ax=axes[0, 1])
    axes[0, 1].set(title="Waiting-time distribution", xlabel="Wait (minutes)", ylabel="Cumulative proportion")
    sns.barplot(data=orders, x="period", y="bill", order=period_order, hue="period",
                palette=period_palette, hue_order=period_order, legend=False,
                errorbar=("ci", 95), seed=42, ax=axes[1, 0])
    axes[1, 0].set(title="Mean bills with 95% bootstrap intervals", xlabel="Service period", ylabel="Mean bill ($)")
    sns.countplot(data=orders, x="day", order=day_order, hue="period", palette=period_palette,
                  hue_order=period_order, ax=axes[1, 1])
    axes[1, 1].set(title="Orders by day", xlabel="Day", ylabel="Order count")
    report.suptitle("Café order exploration — synthetic teaching data", fontsize=16)
    output_dir = Path.cwd() / "outputs"
    output_dir.mkdir(exist_ok=True)
    for extension in ["png", "svg"]:
        destination = output_dir / f"seaborn_cafe_report.{extension}"
        report.savefig(destination, dpi=150, bbox_inches="tight")
        print("Saved:", destination)
    plt.show()

## 16. Practice and troubleshooting

1. Replace the mean bill comparison with a median comparison using `estimator=np.median`. Update its title and axis label.
2. Compare histogram bin widths of $5 and $15. Which patterns remain?
3. Facet the relationship chart by party size. Explain what changes after comparing similarly sized parties.
4. Introduce a missing bill and count missing values before plotting. Decide whether each chart should omit that row or report its absence separately.

| Symptom or interpretation trap | What to check |
| --- | --- |
| Column name cannot be interpreted | Inspect `orders.columns` and supply the correct `data=` |
| Unexpected averaging | `barplot` and `lineplot` aggregate by default |
| Figure appears outside the intended subplot | Use axes-level functions with `ax=` for custom layouts |
| Colors change meaning between panels | Use an explicit palette dictionary and group order |
| Group is absent | Inspect missing values and group counts before plotting |
| Smooth density extends beyond plausible values | Inspect smoothing and boundaries; consider a histogram or ECDF |
| Warning about a palette without hue | Map the category to `hue`; disable a redundant legend if appropriate |

## Further reading

See the official [Seaborn tutorial](https://seaborn.pydata.org/tutorial) for statistical chart design, the [function overview](https://seaborn.pydata.org/tutorial/function_overview.html) for axes-level and figure-level interfaces, and the [histplot reference](https://seaborn.pydata.org/generated/seaborn.histplot.html) for binning and normalization options.

## Explain what the evidence supports

Choose two café groups and compare sample size, center and spread. Describe how a count chart and a mean chart answer different questions about those groups.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: compare latency distributions after an uneven rollout

A service release appears slower in an overall latency dashboard, yet engineers say each route improved. Reconstruct the report from 2,000 generated request events with a changed traffic mix, missing timeout measurements, three malformed values and 20 duplicate retries. Compare pooled and route-specific summaries, retain outcome counts, and control how much detail reaches the final chart. The data is a constructed operational scenario, not a measurement of a real service.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through a misleading pooled average

Suppose fast requests improve from 80 to 60 ms and slow requests from 800 to 600 ms. Before the release, traffic is mostly fast; afterward it is mostly slow. The weighted overall mean can rise even when both conditional means fall. Calculate both levels before drawing a conclusion. Timeouts are absent durations here, not zero milliseconds; report them separately. A causal claim about the release would need a stronger comparison design.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

p2_rng=np.random.default_rng(72);p2_records=[]
for p2_period,p2_route,p2_count,p2_mean in [('before','fast',900,80),('before','slow',100,800),('after','fast',100,60),('after','slow',900,600)]:
    for p2_value in p2_rng.lognormal(np.log(p2_mean)-0.5*0.15**2,0.15,p2_count):
        p2_records.append({'request_id':len(p2_records),'period':p2_period,'route':p2_route,'latency_ms':str(round(p2_value,2)),'status':'ok'})
p2_raw=pd.DataFrame(p2_records)
p2_raw.loc[::97,'latency_ms']='';p2_raw.loc[::97,'status']='timeout'
p2_raw.loc[[11,12,13],'latency_ms']='broken'
p2_raw=pd.concat([p2_raw,p2_raw.iloc[-20:]],ignore_index=True)
p2_unique=p2_raw.drop_duplicates().copy()
p2_unique['latency']=pd.to_numeric(p2_unique.latency_ms,errors='coerce')
p2_bad=(p2_unique.status=='ok')&p2_unique.latency.isna()
p2_observed=p2_unique[(p2_unique.status=='ok')&~p2_bad].copy()
p2_summary=p2_observed.groupby(['period','route']).latency.agg(count='size',median='median',mean='mean',p95=lambda x:x.quantile(.95)).reset_index()
p2_means=p2_summary.pivot(index='route',columns='period',values='mean')
assert (p2_means['after']<p2_means['before']).all()
p2_pooled=p2_observed.groupby('period').latency.mean()
assert p2_pooled['after']>p2_pooled['before']
assert len(p2_unique)==2000 and p2_bad.sum()==3
print({'raw':len(p2_raw),'unique':len(p2_unique),'timeouts':int((p2_unique.status=='timeout').sum()),'malformed_ok_rows':int(p2_bad.sum())})
display(p2_summary)

### Work through counts, uncertainty and distribution shape

Compare a pooled mean chart with route-faceted empirical cumulative distributions. An ECDF shows the fraction of measured successful requests below each latency and avoids choosing histogram bins. The sample sizes differ sharply across groups, so show counts in the accompanying table.

A bootstrap interval around a mean does not repair the wrong grouping or recover timeouts. This example uses 200 bootstrap replicates to bound tutorial work; use a planned uncertainty method and enough replicates for a real report.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
p2_fig,p2_axes=plt.subplots(1,2,figsize=(11,4),layout='constrained')
sns.barplot(data=p2_observed,x='period',y='latency',order=['before','after'],errorbar=('ci',95),n_boot=200,seed=7,ax=p2_axes[0])
p2_axes[0].set(title='Pooled successes mix route populations',ylabel='Mean latency (ms)')
sns.ecdfplot(data=p2_observed,x='latency',hue='period',hue_order=['before','after'],ax=p2_axes[1])
p2_axes[1].set(title='Pooled ECDF still reflects traffic mix',xlabel='Observed latency (ms)')
plt.show()
p2_grid=sns.displot(data=p2_observed,x='latency',hue='period',col='route',kind='ecdf',height=3.2,aspect=1.3)
p2_grid.set_axis_labels('Observed success latency (ms)','Fraction at or below latency')
p2_grid.savefig(p2_out/'seaborn_route_ecdf.png');plt.show()
p2_summary.to_csv(p2_out/'seaborn_release_summary.csv',index=False)
assert p2_summary['count'].sum()==len(p2_observed)

### Explain why the second view is necessary

A faster route-specific distribution reaches a given cumulative fraction at a smaller x value. Read that direction carefully: a taller ECDF at one latency is not automatically worse. Pooled plots answer a different population question and can remain useful for capacity planning, provided their changing composition is explicit.

**Try:** replace the ECDF with a violin plot. **Worked reasoning:** the violin adds smoothing and bandwidth choices that can imply density beyond feasible values. Keep the full-data counts and compare against the ECDF before describing small bumps as real subpopulations. Neither visual treatment solves missing-not-at-random timeout durations.